# Exploratory Data Analysis (EDA)
**Course:** ALY 6130 — Enterprise Risk Management | **Group 1**  
**Company:** Manulife–Mahindra Life Insurance JV (India)  

This notebook explores all datasets before modeling — both the Risk Register and the 4 synthetic market datasets.

> Upload all files to `/content/` in Colab before running:
> - `risk_register_data.xlsx`
> - `irdai_approval_history.csv`
> - `india_insurance_market.csv`
> - `ai_fairness_applicants.csv`
> - `lic_competitor_data.csv`

## Step 1: Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
print('Libraries loaded.')

## Step 2: Load All Datasets

In [ ]:
# 1. Risk Register
df_raw = pd.read_excel('/content/risk_register_data.xlsx')
df_raw.columns = ['Risk_ID','Risk_Description','Likelihood_Score','Impact_Score'] + list(df_raw.columns[4:])
df = df_raw[['Risk_ID','Risk_Description','Likelihood_Score','Impact_Score']].dropna()
df = df[df['Risk_ID'].apply(lambda x: isinstance(x,(int,float)))].copy()
df['Risk_ID'] = df['Risk_ID'].astype(int)
df['Likelihood_Score'] = df['Likelihood_Score'].astype(int)
df['Impact_Score'] = df['Impact_Score'].astype(int)
df['Risk_Score'] = df['Likelihood_Score'] * df['Impact_Score']
def classify_severity(row):
    if row['Likelihood_Score']>=8 and row['Impact_Score']>=5: return 'High'
    elif row['Likelihood_Score']>=5 or row['Impact_Score']>=3: return 'Medium'
    return 'Low'
df['Severity'] = df.apply(classify_severity, axis=1)
cat_map = {
    1:'Competitive',2:'Competitive',3:'Competitive',
    4:'Financial',5:'Financial',6:'Financial',7:'Financial',8:'Financial',
    9:'Operational',10:'Operational',11:'Operational',12:'Operational',13:'Operational',14:'Operational',
    15:'Technology/AI',16:'Technology/AI',17:'Technology/AI',18:'Technology/AI',19:'Technology/AI',20:'Technology/AI',
    21:'Info Security',22:'Info Security',23:'Info Security',24:'Info Security',
    25:'Legal/Compliance',26:'Legal/Compliance',27:'Legal/Compliance',28:'Legal/Compliance',
    29:'Reputational',30:'Reputational',31:'Reputational',32:'Reputational'
}
df['Category'] = df['Risk_ID'].map(cat_map)

# 2. IRDAI Approval History
df_irdai = pd.read_csv('/content/irdai_approval_history.csv')

# 3. India Insurance Market
df_market = pd.read_csv('/content/india_insurance_market.csv')

# 4. AI Fairness Applicants
df_demo = pd.read_csv('/content/ai_fairness_applicants.csv')

# 5. LIC Competitor Data
df_agent = pd.read_csv('/content/lic_competitor_data.csv')

print('All datasets loaded:')
print(f'  Risk Register:        {len(df)} risks')
print(f'  IRDAI Approvals:      {len(df_irdai)} records')
print(f'  Insurance Market:     {len(df_market)} years')
print(f'  AI Fairness Apps:     {len(df_demo)} applicants')
print(f'  Competitor Data:      {len(df_agent)} months')

## Step 3: Risk Register — Score & Severity Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Risk score histogram
axes[0].hist(df['Risk_Score'], bins=15, color='#2E75B6', edgecolor='white', alpha=0.85)
axes[0].axvline(45, color='#C00000', lw=2, ls='--', label='HIGH (≥45)')
axes[0].axvline(20, color='#FF8C00', lw=2, ls='--', label='MEDIUM (≥20)')
axes[0].set_title('Risk Score Distribution (All 32 Risks)', fontweight='bold')
axes[0].set_xlabel('Risk Score (L × I)')
axes[0].set_ylabel('Frequency')
axes[0].legend(fontsize=8)

# Severity pie
sev_colors = {'High':'#C00000','Medium':'#FF8C00','Low':'#375623'}
sev_counts = df['Severity'].value_counts()
axes[1].pie(sev_counts.values, labels=sev_counts.index,
            colors=[sev_colors[s] for s in sev_counts.index],
            autopct='%1.1f%%', startangle=90,
            wedgeprops={'edgecolor':'white','linewidth':2})
axes[1].set_title('Severity Distribution', fontweight='bold')

# Category mean score
cat_mean = df.groupby('Category')['Risk_Score'].mean().sort_values(ascending=True)
axes[2].barh(cat_mean.index, cat_mean.values, color='#1F4E79', edgecolor='white', alpha=0.85)
axes[2].set_title('Mean Risk Score by Category', fontweight='bold')
axes[2].set_xlabel('Mean Risk Score')

plt.suptitle('Risk Register EDA — Manulife–Mahindra JV | ALY 6130 Group 1',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print('Risk Register Summary:')
print(df[['Likelihood_Score','Impact_Score','Risk_Score']].describe().round(2))

## Step 4: IRDAI Approval History — Distribution Fitting

In [ ]:
from scipy import stats

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Fit log-normal to actual data
data = df_irdai['Approval_Timeline_Months'].values
mu_fit = np.mean(np.log(data))
sigma_fit = np.std(np.log(data))
mu_months = np.exp(mu_fit + sigma_fit**2/2)
print(f'Fitted Log-Normal: μ={np.exp(mu_fit):.1f} months, σ={sigma_fit:.3f}')
print(f'Mean approval time: {data.mean():.1f} months')
print(f'P90 approval time: {np.percentile(data,90):.1f} months')

# Histogram with fitted curve
axes[0].hist(data, bins=8, density=True, color='#FF8C00', edgecolor='white', alpha=0.85, label='Actual data')
x = np.linspace(6, 36, 200)
pdf = stats.lognorm.pdf(x, s=sigma_fit, scale=np.exp(mu_fit))
axes[0].plot(x, pdf, color='#C00000', lw=2.5, label='Fitted Log-Normal')
axes[0].axvline(18, color='#1F4E79', lw=2, ls='--', label='Warning threshold (18mo)')
axes[0].set_title('IRDAI Approval Timeline — Data + Fitted Distribution', fontweight='bold')
axes[0].set_xlabel('Approval Timeline (Months)')
axes[0].set_ylabel('Density')
axes[0].legend(fontsize=9)

# Queries received
q_counts = df_irdai['Queries_Received'].value_counts().sort_index()
axes[1].bar(q_counts.index, q_counts.values, color='#2E75B6', edgecolor='white', alpha=0.85)
axes[1].set_title('Regulatory Queries Received per Application', fontweight='bold')
axes[1].set_xlabel('Number of Queries')
axes[1].set_ylabel('Frequency')
axes[1].axvline(df_irdai['Queries_Received'].mean(), color='#C00000', lw=2,
                ls='--', label=f'Mean = {df_irdai["Queries_Received"].mean():.1f}')
axes[1].legend()

plt.tight_layout()
plt.show()

## Step 5: India Insurance Market Trends

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Market share trend
axes[0].plot(df_market['Year'], df_market['LIC_Market_Share_Pct'],
             'o-', color='#C00000', lw=2.5, ms=8, label='LIC')
axes[0].plot(df_market['Year'], df_market['HDFC_Life_Pct'],
             's-', color='#2E75B6', lw=2, ms=7, label='HDFC Life')
axes[0].plot(df_market['Year'], df_market['ICICI_Pru_Life_Pct'],
             '^-', color='#FF8C00', lw=2, ms=7, label='ICICI Pru')
axes[0].plot(df_market['Year'], df_market['SBI_Life_Pct'],
             'D-', color='#375623', lw=2, ms=7, label='SBI Life')
axes[0].set_title('Market Share Trend (2019–2024)', fontweight='bold')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Market Share (%)')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# Market size growth
axes[1].bar(df_market['Year'], df_market['Total_Market_Size_USD_B'],
            color='#1F4E79', edgecolor='white', alpha=0.85)
axes[1].set_title('India Life Insurance Market Size (USD Billion)', fontweight='bold')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Market Size (USD Billion)')
for i, (yr, val) in enumerate(zip(df_market['Year'], df_market['Total_Market_Size_USD_B'])):
    axes[1].text(yr, val+1, f'${val:.0f}B', ha='center', fontsize=9, fontweight='bold')

# LIC growth rate
lic_growth = df_market['LIC_Market_Share_Pct'].pct_change().dropna() * 100
print(f'LIC market share change per year (mean): {lic_growth.mean():.2f}%')
print(f'This confirms our R1 Monte Carlo input: Normal(μ=2.1%, σ=0.8%) competitor growth')

plt.tight_layout()
plt.show()

## Step 6: AI Fairness — Demographic Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Approval rate by type
approval_by_type = df_demo.groupby('Applicant_Type')['AI_Approved'].mean() * 100
bars = axes[0].bar(approval_by_type.index, approval_by_type.values,
                   color=['#2E75B6','#C00000'], edgecolor='white', width=0.5)
axes[0].axhline(80, color='#FF8C00', lw=2, ls='--', label='Target (80%)')
axes[0].set_title('AI Approval Rate:\nUrban vs Rural', fontweight='bold')
axes[0].set_ylabel('Approval Rate (%)')
axes[0].set_ylim(0, 100)
axes[0].legend(fontsize=9)
for bar, val in zip(bars, approval_by_type.values):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
                 f'{val:.1f}%', ha='center', fontweight='bold')

# Parity score
urban_rate = df_demo[df_demo['Applicant_Type']=='Urban']['AI_Approved'].mean()
rural_rate  = df_demo[df_demo['Applicant_Type']=='Rural']['AI_Approved'].mean()
parity = rural_rate / urban_rate
axes[1].bar(['Demographic\nParity Score','IRDAI Threshold'],
            [parity, 0.80], color=['#C00000','#375623'], edgecolor='white', width=0.4)
axes[1].set_title(f'Parity Score vs IRDAI Threshold', fontweight='bold')
axes[1].set_ylabel('Score')
axes[1].set_ylim(0, 1.0)
axes[1].text(0, parity+0.02, f'{parity:.3f}', ha='center', fontweight='bold', color='#C00000')
axes[1].text(1, 0.82, '0.800', ha='center', fontweight='bold', color='#375623')

# Income distribution
urban_inc = df_demo[df_demo['Applicant_Type']=='Urban']['Annual_Income_INR']/1000
rural_inc = df_demo[df_demo['Applicant_Type']=='Rural']['Annual_Income_INR']/1000
axes[2].hist(urban_inc, bins=30, alpha=0.7, color='#2E75B6', label='Urban', density=True)
axes[2].hist(rural_inc, bins=30, alpha=0.7, color='#C00000', label='Rural', density=True)
axes[2].set_title('Income Distribution:\nUrban vs Rural (₹000s)', fontweight='bold')
axes[2].set_xlabel('Annual Income (₹ Thousands)')
axes[2].set_ylabel('Density')
axes[2].legend(fontsize=9)

plt.suptitle('AI Fairness EDA — Demographic Parity Analysis', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Urban approval rate:          {urban_rate:.2%}')
print(f'Rural approval rate:           {rural_rate:.2%}')
print(f'Demographic parity score:      {parity:.3f}')
print(f'IRDAI threshold:               0.800')
print(f'Bias detected (below 0.80):    {"YES ⚠️" if parity < 0.80 else "NO ✅"}')

## Step 7: Competitor Agent Data — R1 Context

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Agent count trend
axes[0].plot(range(len(df_agent)), df_agent['LIC_Agent_Count']/1000,
             color='#C00000', lw=2.5, label='LIC')
axes[0].plot(range(len(df_agent)), df_agent['HDFC_Life_Agent_Count']/1000,
             color='#2E75B6', lw=2, label='HDFC Life')
axes[0].plot(range(len(df_agent)), df_agent['SBI_Life_Agent_Count']/1000,
             color='#375623', lw=2, label='SBI Life')
axes[0].set_title('Agent Network Size (Thousands)\nJan 2022–Dec 2024', fontweight='bold')
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Agents (Thousands)')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# CAC trend
axes[1].plot(range(len(df_agent)), df_agent['Avg_CAC_INR'],
             color='#1F4E79', lw=2.5, marker='o', ms=4)
axes[1].axhline(1800, color='#375623', lw=2, ls='--', label='Mean CAC ₹1,800')
axes[1].axhline(2500, color='#C00000', lw=2, ls='--', label='Warning threshold ₹2,500')
axes[1].set_title('Customer Acquisition Cost (CAC)\nMonthly Trend', fontweight='bold')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('CAC (INR)')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Competitor EDA — Agent Network & CAC | ALY 6130 Group 1',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'LIC mean agent count: {df_agent["LIC_Agent_Count"].mean():,.0f}')
print(f'Mean CAC (INR): {df_agent["Avg_CAC_INR"].mean():,.0f}')
print(f'This confirms our R1 CAC input: LogNormal(μ=₹1,800, σ=₹400)')

## EDA Summary

| Dataset | Key Finding | Impact on Monte Carlo |
|---|---|---|
| Risk Register | 9 HIGH risks, 5 tied at score 56 | Motivates 3D analysis |
| IRDAI Approvals | Mean = 13.8 months, fitted Log-Normal | R9 distribution confirmed |
| Insurance Market | LIC declining 74.8%→60.8% (2019–2024) | R1 competitor growth rate |
| AI Fairness | Parity score 0.711 (below 0.80 threshold) | R15 Beta distribution |
| Competitor Data | Mean CAC ₹1,800, LIC 1.35M agents | R1 agent/CAC inputs |